# P15–P16 — Reproducibilidad, congelamiento e incidencias
## (a) Paso del protocolo y objetivo
**P15:** dejar **evidencia reproducible**: entorno fijado (versiones exactas), semillas, huellas sha256 y el **modelo congelado** (compuerta **G5**: el modelo de las sesiones es *exactamente* el medido en G3, con las respuestas verificadas de G4). **P16:** registrar **toda** desviación o error en `incident_log.csv`; ninguna se resuelve en silencio. Sustentan el **OE2** (la arquitectura es reproducible) y la validez de todos los demás resultados.
**Origen:** huellas y versiones **Reales** (hashes, sin datos); la bitácora y la matriz de trazabilidad son registros del proyecto.

## (b) Código del repositorio que lo implementa
`congelar_modelo.py` (congela y verifica con `--verificar`), `verificar_referencias.py` (contrasta rutas y cifras del protocolo con el repositorio), `auditar_diseno_piloto.py` y `auditar_fecha_corte.py` (auditorías de consistencia documental), `estado_compuertas.py` (tablero) y los scripts de este material (`construir_colab*.py`, `colab_util.py`, `colab_metadatos.py`). Entradas: modelo, dominio, configuración, entrenamiento, umbral; Salidas: `logs/v3_real/modelo_congelado.json`, `incident_log.csv`.

In [1]:
# ----- Preparación mínima (igual en todos los cuadernos) -----
import os, sys, zipfile
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    CARPETA_DRIVE = Path("/content/drive/MyDrive/MPSR_colab")
    BASE = Path("/content/mpsr")
else:
    CARPETA_DRIVE = Path(os.environ.get("MPSR_DRIVE", ".")).resolve()
    BASE = Path(os.environ.get("MPSR_BASE", "mpsr_trabajo")).resolve()
BASE.mkdir(parents=True, exist_ok=True)
ZIP_PUBLICO = CARPETA_DRIVE / "MPSR_colab_publico.zip"
if not ZIP_PUBLICO.exists():
    raise SystemExit(f"Falta {ZIP_PUBLICO}. Sube MPSR_colab_publico.zip a la carpeta MPSR_colab de tu Drive (ver LEEME_colab_por_paso.md).")
if not (BASE / "scripts" / "colab_util.py").exists():          # solo descomprime la primera vez
    with zipfile.ZipFile(ZIP_PUBLICO) as z:
        z.extractall(BASE)
sys.path.insert(0, str(BASE / "scripts"))
import colab_util as U
U.iniciar(BASE, CARPETA_DRIVE)                                  # rutas, paquete privado (si existe) y detección de Rasa
from colab_util import *                                        # rotulo, comparar, sha256, leer_json, tabla_scripts, ic_bootstrap, ...

Listo · BASE = base · paquete privado: SÍ · Rasa: no instalado (no hace falta)


In [2]:
tabla_scripts(scripts_de("12"))

▌ORIGEN: Código del proyecto (no es dato del estudio)


,script,paso del protocolo,propósito (docstring),entradas,salidas,cómo ejecutarlo,usa Rasa,rol en este cuaderno
0,auditar_diseno_piloto.py,P12 (auditoría),"Auditoría de las menciones al diseño anterior del piloto (V1.2: 120 personas, dos visitas, WhatsApp) frente al vigente (V1.3: n = 60, una sesión asistida).",Documentos del repositorio,Menciones al diseño anterior (120 personas) frente al vigente (n = 60),python scripts/auditar_diseno_piloto.py,No,Documentado
1,auditar_fecha_corte.py,2.14 (auditoría),"Auditoría de «fecha de corte» / «fecha límite» / «plazo» frente a las compuertas de avance (protocolo V1.4, sección 2.14).",Documentos del repositorio,Menciones a «fecha de corte» frente a las compuertas,python scripts/auditar_fecha_corte.py,No,Documentado
2,capturas_avance.py,Evidencias de avance (P15),"Capturas (PNG) de las salidas REALES guardadas en los cuadernos y de ejecuciones locales de solo lectura, para la Ficha del Laboratorio Semana 4 (Tesis II).",Cuadernos ejecutados y logs existentes,"evidencias/capturas_avance/ (PNG, INDICE.md, PDF)","python scripts/capturas_avance.py ejecutar-local corre en vivo (solo lectura) las pruebas tests/smoke_*.py, la verificación del modelo congelado y las huellas de archivos protegidos;",No,Documentado
3,colab_metadatos.py,Documentación (Colab),Metadatos CURADOS a mano para el cuaderno de Colab (Colab_Avance_MPSR_v1) y docs/Documentacion_Codigo_v1.md.,—,"Tablas curadas (paso del protocolo, glosario, trazabilidad) que lee construir_colab.py",python scripts/colab_metadatos.py --help,No,Documentado
4,colab_util.py,Documentación (Colab),Utilidades comunes de los cuadernos de Colab por paso (notebooks/colab_por_paso/).,—,"Funciones comunes de los cuadernos por paso: rutas, rótulos, comparar, métricas, tabla de scripts",python scripts/colab_util.py --help,Sí (import),Documentado (se importa en cada cuaderno)
5,congelar_modelo.py,G5,Congela el modelo que se usará en el piloto (protocolo V1.3).,"Modelo, dominio, entrenamiento, configuración, umbral",logs/v3_real/modelo_congelado.json; opción --verificar,python scripts/congelar_modelo.py --modelo models/v3_real/<modelo>.tar.gz,No,Lógica recalculada (etapa 6)
6,construir_colab.py,Documentación (Colab),"Construye el material de Colab del proyecto (Colab_Avance_MPSR_v1): paquetes .zip, cuaderno .ipynb ejecutado, documentación en texto y LEEME.",Repositorio (solo lectura) + tests/,"colab_paquetes/*.zip, notebooks/Colab_Avance_MPSR_v1.ipynb, docs/Documentacion_Codigo_v1.md",python scripts/construir_colab.py --todo --incluir-privado # construye y verifica todo,No,Documentado (genera este cuaderno y los paquetes)
7,construir_colab_por_paso.py,Documentación (Colab),Divide el cuaderno único (Colab_Avance_MPSR_v1) en cuadernos por paso del protocolo + un índice.,Cuaderno único + paquetes,"notebooks/colab_por_paso/*.ipynb, colab_paquetes/colab_por_paso/",python scripts/construir_colab_por_paso.py --etiquetas (lista las etiquetas reutilizables del cuaderno único),No,Documentado (genera los cuadernos por paso)
8,estado_compuertas.py,2.14 (G1–G7),"Tablero de las compuertas de avance (protocolo V1.4, sección 2.14, G1 a G7).",Archivos de evidencia de logs/ y docs/,"logs/avance/estado_compuertas.{json,md}",python scripts/estado_compuertas.py,No,Resultado guardado (etapa 12)
9,evidencia_ejecucion.py,Evidencias de avance (P15),"PDF «Evidencia de ejecución» (pruebas): entrenamiento, métodos comparados, métricas y test, pruebas con datos sintéticos y reales, y el ejecutable final aplicado a 5 personas.",Cuadernos ejecutados y registros de ejecución,evidencias/capturas_avance/Evidencia_Ejecucion.pdf (sin fechas),python scripts/evidencia_ejecucion.py (requiere los .zip ya construidos y las ejecuciones locales de capturas_avance.py),No,Documentado


## (c) Celdas de código
### 1. Entorno y semillas

### Entorno y reproducibilidad
**Qué se hace:** se registran las versiones de Python y de las librerías, las semillas, la carpeta de trabajo, y (en Colab) se monta Drive y se descomprimen los paquetes.
**Por qué:** la reproducibilidad (paso P15 del protocolo) exige poder decir con qué versiones se obtuvo cada cifra. El modelo congelado se entrenó con **Python 3.10.11, Rasa 3.6.21, TensorFlow 2.12.0, scikit-learn 1.1.3, numpy 1.23.5** (`requirements.txt`).
**Qué necesita Rasa y qué no:** Rasa 3.6 exige Python 3.8–3.10 y numpy < 1.24; **el Colab actual trae Python más nuevo**, así que *no se instala Rasa*. Todo lo demás (pandas, scikit-learn, scipy, hashes, tablero, alfa de Cronbach, métricas, bootstrap, McNemar) corre siempre. Lo único que depende de Rasa es **entrenar/cargar el modelo DIET**; para eso el cuaderno usa las **predicciones y resultados guardados** y lo declara en cada celda.

In [3]:
# ----- Versiones, semillas y capacidades -----
versiones = {"Python": platform.python_version(), "Sistema": platform.system() + " " + platform.release(), "numpy": np.__version__, "pandas": pd.__version__}
for mod in ("scipy", "sklearn", "matplotlib", "openpyxl", "yaml"):
    try:
        m = __import__(mod)
        versiones[mod] = getattr(m, "__version__", "?")
    except ImportError:
        versiones[mod] = "NO instalado"
versiones["Rasa"] = VERSION_RASA or "no instalado (no es necesario para este cuaderno)"
rotulo("Entorno de ejecución de este cuaderno (no es dato del estudio)")
display(pd.DataFrame({"componente": list(versiones), "versión aquí": list(versiones.values())}))

ref = leer_json("logs/v3_real/modelo_congelado.json")
rotulo("Entorno con el que se entrenó el modelo congelado (leído de modelo_congelado.json)")
display(pd.DataFrame({"componente": list(ref["versiones"]), "versión del congelamiento": list(ref["versiones"].values())}))
print("Semilla global del proyecto: 42 (DIET, SVM, particiones y bootstrap). Remuestreos del bootstrap: 1000.")
print("Rasa disponible aquí:", HAY_RASA, "| Colab:", EN_COLAB)

▌ORIGEN: Entorno de ejecución de este cuaderno (no es dato del estudio)


,componente,versión aquí
0,Python,3.10.11
1,Sistema,Windows 10
2,numpy,1.23.5
3,pandas,2.0.3
4,scipy,1.10.1
5,sklearn,1.1.3
6,matplotlib,3.5.3
7,openpyxl,3.1.5
8,yaml,6.0.3
9,Rasa,no instalado (no es necesario para este cuaderno)


▌ORIGEN: Entorno con el que se entrenó el modelo congelado (leído de modelo_congelado.json)


,componente,versión del congelamiento
0,rasa,3.6.21
1,scikit-learn,1.1.3
2,pandas,2.0.3
3,numpy,1.23.5
4,scipy,1.10.1
5,tensorflow,2.12.0


Semilla global del proyecto: 42 (DIET, SVM, particiones y bootstrap). Remuestreos del bootstrap: 1000.
Rasa disponible aquí: False | Colab: False


### Cómo leer el resultado de esta sección
- La primera tabla dice **dónde corre este cuaderno**; la segunda, **con qué se entrenó el modelo congelado**. Si `scikit-learn`/`numpy` difieren, los números de métricas recalculadas (que dependen solo de fórmulas) no cambian, pero **no se puede cargar un modelo `.joblib`/`.tar.gz` entrenado con otra versión**: por eso no se carga ningún modelo en el cuaderno.
- «Rasa disponible: False» es lo esperado en Colab. Las etapas que normalmente usarían Rasa muestran en su lugar los resultados guardados (etapas 6 y 7).

### 2. Las versiones de `requirements.txt` coinciden con las del congelamiento (Real)

In [4]:
fz = leer_json("logs/v3_real/modelo_congelado.json")
req = {}
for l in (BASE / "requirements.txt").read_text(encoding="utf-8").splitlines():
    m = re.match(r"^([A-Za-z0-9_.-]+)==([\w.]+)", l.strip())
    if m:
        req[m.group(1).lower()] = m.group(2)
rotulo(ORIGEN_REAL, "versiones del modelo congelado frente a requirements.txt")
filas = []
for pkg, v in fz["versiones"].items():
    filas.append({"paquete": pkg, "congelamiento": v, "requirements.txt": req.get(pkg.lower()), "coincide": req.get(pkg.lower()) == v})
    comparar("P15", f"versión de {pkg}", req.get(pkg.lower()), v, "requirements.txt frente a modelo_congelado.json", tipo="txt")
display(pd.DataFrame(filas))
print("Python del congelamiento:", fz["python"], "| commit declarado:", fz["commit"])

▌ORIGEN: Real — versiones del modelo congelado frente a requirements.txt


,paquete,congelamiento,requirements.txt,coincide
0,rasa,3.6.21,3.6.21,True
1,scikit-learn,1.1.3,1.1.3,True
2,pandas,2.0.3,2.0.3,True
3,numpy,1.23.5,1.23.5,True
4,scipy,1.10.1,1.10.1,True
5,tensorflow,2.12.0,2.12.0,True


Python del congelamiento: 3.10.11 | commit declarado: 98d549a-modificado


### 3. Congelamiento e integridad del modelo (G5)

### Congelamiento y verificación de integridad (G5)
**Qué se hace:** se **recalcula el sha256** de cada componente del modelo congelado (modelo, configuración, dominio, conjunto de entrenamiento y umbral) y se compara con `modelo_congelado.json`. Se muestra «intacto» o la discrepancia.
**Por qué:** G5 exige que el modelo con el que se hacen las sesiones sea **exactamente** el medido en G3 (y que las respuestas verificadas en G4 ya formen parte de él). Si un solo byte cambia, el congelamiento se invalida.
**Paso del protocolo:** G5 (antes de la primera sesión del pre-piloto).

In [5]:
fz = leer_json("logs/v3_real/modelo_congelado.json")
previo = leer_json("logs/v3_real/lote2_congelado_previo.json")
rutas = {"modelo": PRIV / "artefactos_congelados/LOTE2-FINAL.tar.gz", "config": BASE / "configs/rasa_config_lote2.yml", "dominio": BASE / "domain_v3.yml",
         "corpus": PRIV / "artefactos_congelados/entrenamiento_lote2.csv", "umbral": BASE / "logs/v3_real/lote2_umbral_congelado.json"}
filas, hay_dif, no_verificables = [], False, 0
for k, esperado in fz["sha256"].items():
    p = rutas[k]
    if p.exists():
        calc = sha256(p)
        estado = "intacto" if calc == esperado else "DISCREPANCIA"
        hay_dif |= calc != esperado
        comparar("G5", f"sha256 {k}", calc, esperado, "logs/v3_real/modelo_congelado.json", tipo="txt")
    else:
        calc, estado = "—", "no verificable aquí (requiere paquete privado)"
        no_verificables += 1
    filas.append({"componente": k, "sha256 congelado (12)": esperado[:12], "sha256 recalculado (12)": calc[:12], "estado": estado, "igual al congelamiento previo al lote 2": previo["sha256"].get(k) == esperado})
rotulo(ORIGEN_REAL, "huellas del modelo congelado LOTE2-FINAL v1 (hashes, sin datos)")
display(pd.DataFrame(filas))
if hay_dif:
    print("¡¡DISCREPANCIA!! Al menos un componente cambió: el congelamiento NO está intacto.")
elif no_verificables:
    print(f"Sin discrepancias en los componentes verificados; {no_verificables} componente(s) requieren el paquete privado.")
else:
    print("Congelamiento: INTACTO (los 5 componentes coinciden).")
um = leer_json("logs/v3_real/lote2_umbral_congelado.json")
print("Umbral congelado: t =", um["t"], "| ambigüedad =", um.get("ambiguity_threshold"), "| versión:", fz["version"])
print("Frases de entrenamiento declaradas:", fz["frases_entrenamiento"], "| Python/Rasa del congelamiento:", fz["python"], "/", fz["versiones"]["rasa"])
comparar("G5", "umbral t", um["t"], 0.5, "Protocolo V1.8", tol=0)
comparar("G5", "frases de entrenamiento", fz["frases_entrenamiento"], 943, "Protocolo V1.8 (707 + 185 + 51)", tol=0)

▌ORIGEN: Real — huellas del modelo congelado LOTE2-FINAL v1 (hashes, sin datos)


,componente,sha256 congelado (12),sha256 recalculado (12),estado,igual al congelamiento previo al lote 2
0,modelo,be1a5ad06357,be1a5ad06357,intacto,True
1,config,e975783f2bcd,e975783f2bcd,intacto,True
2,dominio,7a69ac91c32c,7a69ac91c32c,intacto,True
3,corpus,c035252d63aa,c035252d63aa,intacto,True
4,umbral,410d9898240a,410d9898240a,intacto,True


Congelamiento: INTACTO (los 5 componentes coinciden).
Umbral congelado: t = 0.5 | ambigüedad = 0.1 | versión: LOTE2-FINAL v1 (DIET 100/64/20, seed 42, 943 frases de entrenamiento, umbral t=0,50)
Frases de entrenamiento declaradas: 943 | Python/Rasa del congelamiento: 3.10.11 / 3.6.21


### Cómo leer el resultado de esta sección
- **«intacto»** en los cinco componentes = el modelo, el dominio (las 54 respuestas), la configuración, el entrenamiento (943 frases) y el umbral son exactamente los congelados. Los 12 primeros caracteres del hash bastan para una lectura visual; la comparación usa los 64.
- La columna «igual al congelamiento previo al lote 2» confirma que se congeló **antes de abrir el lote 2** (`lote2_congelado_previo.json`) y no se reentrenó después de ver los resultados.
- Con solo el paquete público se verifican dominio, configuración y umbral; el **modelo** (33 MB) y el **entrenamiento** viajan en el paquete privado. Esto replica la opción `--verificar` de `scripts/congelar_modelo.py`, sin cargar el modelo (no hace falta Rasa).

### 4. P16 — Bitácora de incidencias

### Bitácora de incidencias (P16)
**Qué se hace:** se lee `incident_log.csv` y se muestra como tabla: fecha, incidencia, decisión y justificación, más dos columnas **orientativas** (`tipo` técnica/metodológica y `afecta validez`).
**Por qué:** el protocolo manda registrar **toda** desviación; ninguna se resuelve en silencio. La bitácora es la evidencia de que los cambios y errores quedaron declarados.
**Aviso:** el archivo original **no** tiene las columnas «tipo» ni «afecta validez»; aquí se *infieren por palabras clave* solo para orientar la lectura. Son una heurística, no una clasificación del tesista.

In [6]:
inc = pd.read_csv(BASE / "incident_log.csv", dtype=str, keep_default_na=False, encoding="utf-8")
inc.columns = ["fecha", "incidencia", "descripción", "decisión", "justificación"]
PAL_TECNICA = r"bloque|smart app|dll|\.pyd|versi[oó]n de|dependencia|entorno|sklearn|scikit|tensorflow|rasa\.exe|windows|encoding|heredoc|script|error de|fix|ejecutable"
PAL_METODOL = r"fuga|test|evaluaci|umbral|lote|compuerta|congel|etiqueta|revisi|kappa|g[1-7]\b|muestra|dise[ñn]o|frase|protocolo|corpus|partici|desviaci"
PAL_VALIDEZ = r"fuga|evaluaci[oó]n|test|frases? (real|del lote)|umbral|congel|duplicad|g3|kappa|contamin|sesgo|sin revisar|no comparable|artefacto"
txt = (inc["incidencia"] + " " + inc["descripción"]).str.lower()
tec, met = txt.str.contains(PAL_TECNICA, regex=True), txt.str.contains(PAL_METODOL, regex=True)
inc["tipo (heurística)"] = np.where(tec & met, "mixta", np.where(tec, "técnica", np.where(met, "metodológica", "sin clasificar")))
inc["afecta validez (heurística)"] = np.where(txt.str.contains(PAL_VALIDEZ, regex=True), "posible: revisar", "no declarado")
mostrar = inc.assign(descripción=inc["descripción"].str.slice(0, 170), decisión=inc["decisión"].str.slice(0, 120))[["fecha", "tipo (heurística)", "afecta validez (heurística)", "incidencia", "descripción", "decisión"]]
rotulo("Bitácora del proyecto (registro de decisiones, no datos del estudio)", f"{len(inc)} incidencias")
print("Incidencias:", len(inc), "| por tipo (heurística):", inc["tipo (heurística)"].value_counts().to_dict(), "| con posible efecto en la validez:", int((inc["afecta validez (heurística)"] != "no declarado").sum()))
display(mostrar)
rotulo("Últimas 5 incidencias, con su justificación completa")
for _, r in inc.tail(5).iterrows():
    print(f"- {r['fecha']} · {r['incidencia']}\n    Decisión: {r['decisión'][:400]}\n    Justificación: {r['justificación'][:400]}")

▌ORIGEN: Bitácora del proyecto (registro de decisiones, no datos del estudio) — 72 incidencias
Incidencias: 72 | por tipo (heurística): {'metodológica': 47, 'mixta': 12, 'sin clasificar': 7, 'técnica': 6} | con posible efecto en la validez: 33


,fecha,tipo (heurística),afecta validez (heurística),incidencia,descripción,decisión
0,2026-10-01,sin clasificar,no declarado,N/A (planificación piloto),La MPSR no otorga permiso para desplegar el chatbot en su plataforma ni acceso físico al local (ni siquiera cartel/QR),Rediseñar P01/P11.2/P12/P13: despliegue en canal propio (WhatsApp/Telegram/web) y reclutamiento en estudios contables/ju
1,2026-10-01,metodológica,posible: revisar,N/A (diseño metodológico),"No es logísticamente viable recontactar a los mismos sujetos de P01 para el post-test (muestreo en estudios contables/jurídicos, sin canal de seguimiento)",Se cambia el diseño de pre-test/post-test de un solo grupo a dos grupos independientes no equivalentes (comparación está
2,2026-10-02,metodológica,no declarado,N/A (resolución de diseño),"Se contrastó el archivo Encuestas_simuladas_TramiFacil_MPSR_120_v2.xlsx (n=120, generado por el tesista) contra el protocolo vigente que tenía n=30","Se revierte a n=120 (fórmula de poblaciones finitas + análisis de sensibilidad), manteniendo diseño de un solo grupo par"
3,2026-10-02,sin clasificar,no declarado,N/A (instrumento),"Se identificó que TramiFácil es una herramienta de uso interno del personal de la MPSR, no un canal que los ciudadanos puedan usar directamente",Se corrigió la Ficha de Diagnóstico P01 (se eliminó la opción de canal TramiFácil/Ambos y el ítem Likert asociado) y el
4,2026-09-29,técnica,no declarado,"N/A (entorno, P15)","requirements.txt pedía scikit-learn>=1.3 y rasa sin versión; Rasa 3.6 exige scikit-learn<1.2 y numpy<1.24, y pip retrocedía a versiones antiguas de Rasa que no compilan (","Se fijaron versiones verificadas: rasa==3.6.21, scikit-learn==1.1.3, pandas==2.0.3, numpy==1.23.5, scipy==1.10.1 (Python"
5,2026-09-29,técnica,no declarado,"N/A (entorno, P15)",Windows Smart App Control bloquea el ejecutable no firmado venv/Scripts/rasa.exe,Todos los comandos de Rasa se ejecutan como 'python -m rasa ...'; no se modificó ninguna configuración de seguridad del
6,2026-09-29,metodológica,no declarado,P08 (configs/rasa_config.yml),"batch_size: [64, 128] en Rasa no prueba dos valores: hace crecer el lote linealmente de 64 a 128 durante el entrenamiento, lo que no corresponde a la grilla batch_size ∈",batch_size pasa a ser un valor único por corrida; scripts/run_rasa_grid.py genera un config por combinación (logs/<ID>/c
7,2026-10-02,mixta,no declarado,N/A (experiments/*.py),"Los scripts de experiments/ tenían rutas absolutas de otro sistema (/home/claude/corpus_build, /mnt/user-data/uploads) y no corrían en el equipo del tesista",Se reemplazaron solo las rutas por rutas relativas a la raíz del repositorio; no se tocó la lógica
8,2026-10-02,metodológica,no declarado,P05 (corpus starter),"La partición del corpus starter es 162/81/81 (50/25/25), no 70/15/15 como fija el protocolo","Se mantiene la partición recibida para las corridas preliminares (BASE-*, RASA-*); pendiente decidir con la asesora si s"
9,2026-10-02,metodológica,no declarado,P02/P03 (corpus starter),"El corpus starter tiene 54 intenciones; el protocolo (sección 2.3, T03) declara 50 intenciones en 9 categorías",Pendiente: actualizar el protocolo a 54 intenciones o fusionar/eliminar 4 intenciones


▌ORIGEN: Últimas 5 incidencias, con su justificación completa
- 2026-10-09 · Evaluación única del lote 2 (G3): F1 macro 0,9072 y limitación del script de evaluación
    Decisión: No se repite la evaluación ni se cambia el umbral ni el modelo; las limitaciones quedan declaradas en logs/avance/eval_lote2_notas.md y se reportan junto con el resultado; los casos mal clasificados se revisan solo por id y con aprobación del tesista
    Justificación: El resultado es más alto que el estimado por validación cruzada del lote 1 (0,787–0,799); las 56 situaciones son las mismas del lote 1 (independiente por personas, no por situaciones), el entrenamiento incluye frases reales del lote 1 y 13 frases idénticas al entrenamiento se excluyeron del test; leer con cautela y como medición en este diseño, no como generalización a situaciones nuevas
- 2026-10-09 · Congelamiento del modelo medido en el lote 2 (G5 real) y protocolo V1.7 / Nota v10
    Decisión: G5 se evaluó solo con hashes coincidentes y fech

### Cómo leer el resultado (cuaderno 12)
- Cada fila es una desviación o un error **declarado**: por ejemplo, que se mostró por descuido una frase del lote 2 en una salida (y no se usó para ajustar nada), que la evaluación del lote 2 tiene un artefacto en la sección «umbral», o que el registro del pre-piloto se rehízo porque se había mezclado con filas generadas por un asistente.
- «posible: revisar» no quiere decir que la validez esté comprometida; marca las incidencias que tocan evaluación, umbral, congelamiento o fuga, para que el docente las lea con más cuidado.

### 5. Matriz de trazabilidad P01–P16 / G1–G7
La celda siguiente necesita el tablero de compuertas.

In [7]:
tb = leer_json("logs/avance/estado_compuertas.json")

### Matriz de trazabilidad
**Qué se hace:** una tabla única que enlaza cada **paso (P01–P16)** y cada **compuerta (G1–G7)** con el script que lo ejecuta, el archivo de evidencia, el resultado y su estado (*ejecutado / simulado / planificado / pendiente*).
**Por qué:** permite al docente seguir un hilo desde el protocolo hasta el dato. El estado de las compuertas se toma **del tablero**, no se escribe a mano.

In [8]:
tp = pd.DataFrame(meta["traza_pasos"], columns=["id", "paso", "script(s)", "archivo de evidencia", "resultado", "estado"])
tc = []
for gid, nombre, scr, evid in meta["traza_compuertas"]:
    c = next(x for x in tb["compuertas"] if x["id"] == gid)
    estado = {"Cumplida": "ejecutado", "No cumplida": "ejecutado (no cumplida)"}.get(c["estado"], "pendiente")
    if gid == "G7":
        estado = "pendiente (solo demostración simulada)"
    tc.append([gid, f"Compuerta: {nombre}", scr, evid, c["nota"][:230], estado])
tc = pd.DataFrame(tc, columns=tp.columns)


def en_paquete(ev):
    ruta = ev.split(",")[0].split(" + ")[0].strip()
    ruta = ruta.replace("…", "").replace("*", "")
    return "sí" if (BASE / ruta).exists() else "no (privado o ignorado por Git)"


matriz = pd.concat([tp, tc], ignore_index=True)
matriz["en este paquete"] = matriz["archivo de evidencia"].map(en_paquete)
rotulo("Trazabilidad del proyecto (el rótulo de origen de cada dato está en cada cuaderno por paso)")
print("Estados:", matriz["estado"].value_counts().to_dict())
display(matriz)

▌ORIGEN: Trazabilidad del proyecto (el rótulo de origen de cada dato está en cada cuaderno por paso)
Estados: {'ejecutado': 19, 'simulado': 2, 'pendiente': 2, 'ejecutado (no cumplida)': 1, 'pendiente (solo demostración simulada)': 1}


,id,paso,script(s),archivo de evidencia,resultado,estado,en este paquete
0,P01,Diagnóstico y línea base de tiempos,simular_P14_n120.py,corpus/Encuestas_simuladas_TramiFacil_MPSR_120_v2.xlsx,Línea base SIMULADA (n = 120) del diseño V1.2; no es dato de campo,simulado,sí
1,P02,Construcción del corpus,"expand_corpus.py, apply_ampliacion_v3.py",corpus/corpus_metadata.csv,"708 frases sintéticas, 54 intenciones, 9 categorías",ejecutado,sí
2,P03,Auditoría del corpus,audit_corpus.py,"logs/audit_report.txt, corpus/corpus_audit.csv",Duplicados y desbalance auditados,ejecutado,sí
3,P04,Control de fuga por paráfrasis,audit_corpus.py,corpus/corpus_summary.json,leaks_detected = 0 en la partición v3,ejecutado,sí
4,P05,Partición train/validation/test,"split_corpus.py, split_corpus_v3.py, split_lote2.py",corpus/dataset_split.csv; corpus/v3_real; corpus/v3_lote2,546/81/81 sintético; lote 1 real 707/71/114; lote 2 solo test (267),ejecutado,no (privado o ignorado por Git)
5,P06,Normalización (jerga) y formato Rasa,"common.py, export_rasa_nlu.py",configs/jerga_local.csv; data/*.yml,Corpus exportado a formato Rasa,ejecutado,no (privado o ignorado por Git)
6,P07,Línea base TF-IDF + SVM,"train_baseline_p07.py, train_baseline.py","logs/baseline_validation.csv, logs/baseline_test.csv",SVM C = 10 elegido en validación,ejecutado,sí
7,P08,Rasa NLU / DIET (grilla),"run_rasa_grid.py, eval_real.py",logs/v3_real/RASA-*,"Grilla epochs {100,150,200} × batch {64,128} × dim {20,50}",ejecutado,no (privado o ignorado por Git)
8,P09,Respuestas del dominio,"aplicar_tupa.py, aplicar_respuestas.py, humo_respuestas.py",domain_v3.yml; docs/tupa/Verificacion_TUPA_v10_4.xlsx,"44 respuestas de trámites, 0 con [Verificar]",ejecutado,no (privado o ignorado por Git)
9,P10,Repeticiones con 5 semillas,"run_rasa_grid.py, train_baseline.py",logs/v3_real/RASA-e100-b64-d20-s10…s50,"Semillas 10, 20, 30, 40, 50",ejecutado,no (privado o ignorado por Git)


### Cómo leer el resultado de esta sección
- **ejecutado:** el paso se hizo con datos del proyecto. **simulado:** solo se demostró con datos inventados (línea base P01, análisis P14). **pendiente:** depende de datos que aún no existen (piloto de 60 sesiones, G7).
- «en este paquete = no» significa que la evidencia vive en una carpeta ignorada por Git por traer datos de personas; los números agregados de esa evidencia están en los cuadernos 04, 05, 10 y 11.

In [9]:
cierre("P15–P16 Reproducibilidad e incidencias")

P15–P16 Reproducibilidad e incidencias — recalculado frente a reportado: 13 cifras · coinciden: 13 · NO coinciden: 0


,paso,cifra,recalculado,reportado,fuente de lo reportado,coincide
0,P15,versión de rasa,3.6.21,3.6.21,requirements.txt frente a modelo_congelado.json,Sí
1,P15,versión de scikit-learn,1.1.3,1.1.3,requirements.txt frente a modelo_congelado.json,Sí
2,P15,versión de pandas,2.0.3,2.0.3,requirements.txt frente a modelo_congelado.json,Sí
3,P15,versión de numpy,1.23.5,1.23.5,requirements.txt frente a modelo_congelado.json,Sí
4,P15,versión de scipy,1.10.1,1.10.1,requirements.txt frente a modelo_congelado.json,Sí
5,P15,versión de tensorflow,2.12.0,2.12.0,requirements.txt frente a modelo_congelado.json,Sí
6,G5,sha256 modelo,be1a5ad06357b26370fc14f780f49cbf205b08dbe5fe3af383625e44f64b10d9,be1a5ad06357b26370fc14f780f49cbf205b08dbe5fe3af383625e44f64b10d9,logs/v3_real/modelo_congelado.json,Sí
7,G5,sha256 config,e975783f2bcd6dac366b2ff53677c5221ef77044ec23fb615c3d763dcadbf9e4,e975783f2bcd6dac366b2ff53677c5221ef77044ec23fb615c3d763dcadbf9e4,logs/v3_real/modelo_congelado.json,Sí
8,G5,sha256 dominio,7a69ac91c32cf439e82b4c34a553ec066a71d4041b1b8632288864d36a7756b5,7a69ac91c32cf439e82b4c34a553ec066a71d4041b1b8632288864d36a7756b5,logs/v3_real/modelo_congelado.json,Sí
9,G5,sha256 corpus,c035252d63aa89e11295279b363c98d66b9f2e608008b82390b15ce40a6a4adc,c035252d63aa89e11295279b363c98d66b9f2e608008b82390b15ce40a6a4adc,logs/v3_real/modelo_congelado.json,Sí


DISCREPANCIAS (recalculado ≠ reportado): ninguna
Rasa disponible: False · paquete privado: True


## (d) Cómo leer el resultado
- El entorno del congelamiento es Python 3.10.11 con Rasa 3.6.21, TensorFlow 2.12.0, scikit-learn 1.1.3, pandas 2.0.3, numpy 1.23.5 y scipy 1.10.1; `requirements.txt` coincide con esas versiones.
- **«intacto»** en los cinco componentes = el modelo, el dominio (54 respuestas), la configuración, el entrenamiento (943 frases) y el umbral son exactamente los congelados; el congelamiento es **anterior** a abrir el lote 2 y no se reentrenó después de medir.
- La **bitácora** declara cada desviación (por ejemplo, una frase del lote 2 mostrada por error en una salida, el artefacto de la sección «umbral», el registro del pre-piloto rehecho y una prueba de humo que reescribió archivos derivados con el mismo contenido). La columna «afecta validez» es una **heurística** por palabras clave, no una clasificación del tesista.
- La **matriz** une cada paso y compuerta con su script, su evidencia y su estado.

## Limitaciones
- Sin el paquete privado no se pueden verificar el modelo ni el entrenamiento (se muestran las huellas guardadas).
- El commit declarado en el congelamiento termina en «-modificado» (había cambios sin confirmar al congelar); lo que garantiza la integridad son las huellas sha256, no el commit.